# Lab 1 — Train a Decision Model to Play Pac-Man

Follow Kev's published **LoRA plus pointer-head** architecture. First train a fresh decision model from `Qwen/Qwen3.5-4B-Base` on the frozen `decision-v7` training suite. Continue through separate dates/missing-evidence, documents, and skills/devtools stages, then fine-tune your own adapter and head on Pac-Man labels. Original base matrices stay frozen, and LoRA changes the encoder's effective features. This is initial decision-model training over an already pretrained LLM.

**You train Pac-Man. Four ghosts use the classic arcade engine.** The faithful browser recreation supplies the classic maze, original renderer/font/sounds, power pellets, fruit, tunnels, lives and chase/scatter phases. Kev chooses player directions; upstream code controls Blinky, Pinky, Inky and Clyde.

The lab has **one assessed checkpoint: CP1 — fine-tune and evaluate Kev on Pac-Man game states**. Interactive play with the trained adapter is a separate, ungraded activity. The 90-minute route is 0–20 overview and interactive play; 20–30 inspect planning labels; **30–60 mandatory fine-tuning**; 60–80 evaluate; 80–90 explain and submit the checkpoint evidence. Installation, downloads, label generation and the four general-decision training stages are prework.

This notebook runs in **ordinary Jupyter on one laboratory RTX A6000**. The user reports about 20 GB of VRAM use for the current run, which fits one 48 GB card. It uses a separate locked Python 3.13 training environment. BF16 and optimized CUDA kernels remain required. Stage memory checks record the actual usage; this change has been checked on CPU and has not been run on the lab GPU by the author.

The notebook separates **Stage 1: initial decision training → Stage 2: dates/missing evidence → Stage 3: documents → Stage 4: skills/devtools → Stage 5: Pac-Man fine-tuning**. Each saves its own checkpoint and training logs. Calibration is a separate probability-fitting procedure and is not applied here; these fresh runs do not inherit the released model's benchmark scores or fitted temperature. [Recipe and compute](https://github.com/jaredpalmer/kev/blob/84847f0a883d900f7de5b7a57eaa341ca7f9a6b4/docs/model-cards/kev-4b.md#training-procedure).

## Game provenance

Game source: [masonicGIT/pacman](https://github.com/masonicGIT/pacman), GPL-3.0, pinned `7407174c1d6a38be8cd230577489e39e0873145b`. Its source/assets are unchanged; the author documents small [accuracy differences](https://github.com/masonicGIT/pacman#accuracy). Browser and CPU evaluation execute this same arcade engine. Model source: [jaredpalmer/kev](https://github.com/jaredpalmer/kev). The model receives structured state, not screenshots. Inference stays inside the GPU runtime. The board identifies the active adapter, stage, steps, rank and checkpoint hashes.

## 实验室普通 Jupyter：先准备运行环境

从你的本地电脑 SSH 登录实验室，按照 [LAB_SETUP.md](../LAB_SETUP.md) 创建 Python 3.13 Conda 环境 `/chronos_data/conda_envs/qplus-jupyter-py313`，选择 **QPlusLearning (A6000, Python 3.13)** kernel。

本 notebook 默认使用物理 GPU 1、源码 `/chronos_data/huixu/QPlusLearning/labs/lab-01-kev-pacman` 和持久目录 `/chronos_data/huixu/qpluslearning-runtime-a6000`。运行前可设置 `QPLUS_GPU`、`QPLUS_LAB_SOURCE`、`QPLUS_LAB_DIR`。GPU 必须在第一次 CUDA 初始化之前选择；换卡时重启 kernel。

Jupyter 与训练器使用独立环境。`runtime.setup()` 会在 `/chronos_data/conda_envs/qplus-kev-training-py313` 建立 uv 环境（可用 `QPLUS_TRAINING_ENV` 指定），按 Kev 上游 lockfile 安装 Python 3.13 / Torch 2.8.0 / CUDA 12.8，再校验 GPU/BF16、CUDA backward 和优化内核。不要把训练环境指向 Jupyter Conda 环境，也不需要在 kernel 中手动安装 Torch。

本地编译优先复用上面已有的 Jupyter Conda 环境里的 CUDA 12.8 `nvcc` 和开发头文件；不新建第三个 CUDA toolkit 环境。`QPLUS_CUDA_HOME` 可指定已有的其他 CUDA 12.8 toolkit。`nvidia-smi` 显示 CUDA 12.8 不能证明 `nvcc` 已安装；开发工具只在原 wheel 与系统 GLIBC 不兼容、必须本地编译时使用。

按你提供的当前运行约 20 GB 显存占用，单张约 48 GB 的 A6000 足够，本 notebook 默认仅使用 GPU 1。这个占用由你提供，并非本次修改的 GPU 实测结果；不同阶段的占用以各自显存报告为准。所有阶段使用 batch 1 × accumulation 8、row_budget 2048，保持有效 batch 八，每阶段开始前保留长样本显存检查。历史 reference 路径的约 90 GiB OOM 记录不作为当前运行的显存需求；本版本要求 FLA/causal-conv1d 优化内核。

In [1]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, os, sys
GPU = os.environ.get('QPLUS_GPU', '1')
if not GPU.isdecimal():
    raise ValueError('QPLUS_GPU must select one physical GPU index, such as 1')
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
LAB_SOURCE = Path(os.environ.get('QPLUS_LAB_SOURCE', '/chronos_data/huixu/QPlusLearning/labs/lab-01-kev-pacman')).expanduser().resolve()
TRAINING_PROFILE = 'memory_safe'  # Lab execution: batch 1 x accumulation 8, row_budget 2048
LAB_DIR = Path(os.environ.get('QPLUS_LAB_DIR', '/chronos_data/huixu/qpluslearning-runtime-a6000')).expanduser().resolve()
if LAB_DIR.is_relative_to(LAB_SOURCE) or LAB_SOURCE.is_relative_to(LAB_DIR):
    raise ValueError('Use a persistent runtime directory outside the lab source directory')
LAB_DIR.mkdir(parents=True, exist_ok=True)
for name, folder in {'HF_HOME': 'huggingface', 'UV_CACHE_DIR': 'uv', 'UV_PYTHON_INSTALL_DIR': 'python',
                     'PIP_CACHE_DIR': 'pip', 'TORCH_HOME': 'torch', 'TRITON_CACHE_DIR': 'triton',
                     'CUDA_CACHE_PATH': 'cuda'}.items():
    os.environ.setdefault(name, str(LAB_DIR / 'cache' / folder))
COURSE_REVISION = 'e88bf455de872d19861586a90813640b407b9ddc'
FILES = {'api_client.py': '0e48a2dfb4e3bb6d07228341c08cde6c92bf25bc311c91bfcae4bcbf19a8d51e', 'pacman_lab.py': 'aaa36ffc056bb060989f50ba31ebda1a1ae04f2ad6c5d4ace0792941763b97e2', 'cloud_runtime.py': '34df43797749b2225425a52a18963890d80934368ddf18b2085375b5079ad4ba', 'training_monitor.py': '816f46da63ab7e936d55e85425ab9213145f2276fe6f2a260248307b45fbe681', 'lora_recovery.py': 'd01bc4f336b17948dc89fa8e931761c766fed979349a310d19f23e5ce0d6e274', 'checkpoint_backup.py': '6df69d0ecf887026c22573d0477d780aadbed8a8d3927d8d17bfede9ddd55056', 'optimized_training.py': '5c33fac17ec6ad6439193737aea650f85cdfb4445a0d346bb5bed10b858fcdfa', 'training-kernels.json': '24dff024433f61c4a571f16326f4f16c5666297e6514089e66e721b39765e56c', 'training_stages.py': '96fdd59285b8872b946523a88d95393decb73536322ca35e8f67f79cb2a7ef5c', 'training-stages.json': '9be232b730a9550a01011091b1abf3d79eaba3de8db5bdd455783fe5ae0e8ad0', 'games/arcade-engine.js': '3ba2e4889dc1f8ac06ae3723d7419366fa29117449e2f6388d00fc975ddffde4', 'games/arcade-worker.js': 'acd16571bd330e2dc8033ab7c59977f192ae634d1eec4291db7e16a09b730daa', 'games/arcade-browser.js': '74d6ce446b444b983a8f259c4a3fb2afc3885fc73a2abefbcdb89975a6091396', 'games/arcade-shell.html': 'b2606cdd679242c434570fd3850618041f79a8590f3b813106536c4d78341a0c', 'vendor/arcade-pacman/source.json': 'a0f21cf6009b7f7ca195fb8ce5f0730609767e7ced1c9b8cf29152f5bbe52545', 'vendor/arcade-pacman/source.zip': 'fd9ae0cbeebdfd798c609464e884f6e9c9b59673bd50083fbd0e32ea6c26a6e8', 'planner_data.py': '9b325f71bdc08419014b6e69b1541868d483c20efaaf79964892ac3887e65ca4', 'games/arcade-planner.js': 'f505a2b131465f0f170e16731863efe870e8549b50239f41d5a230dd57b7ff57', 'data/pacman-planner-v1.zip': 'a6f7e5282e888ede6a24f28cc7dffdb7d3cf51776ec5c9d6adefcf6e15e9e6a1', 'data/pacman-planner-v1-manifest.json': '3c22ede98985264b46410dc14026eaa22301b79a3f9061cf5063c739fc39b6e2', 'data/pacman-planner-v1-quality.json': 'db6ae0d8b730c7582f3f76e0c0177061d5449cfd55bc87ae3dd4c66277d54fc0'}
LOCAL_RUNTIME_SHA256 = 'c4bda5336d1b6209a1e95c207b3c4391dc0f913c080c3c6275ad7adb50ce37b4'
LOCAL_MEMORY_CHECK_SHA256 = '4385acbd6bc6cced744417e4ecf7fd10575d770318aa16e17e50e46fc7ab45ec'
LOCAL_NATIVE_KERNELS_SHA256 = 'de39291c261fca159a5581232efec8c41f9cdc86a060efb289780ba5d7f16c08'
LOCAL_FILES = {**FILES, 'lab_runtime.py': LOCAL_RUNTIME_SHA256, 'lab_memory_check.py': LOCAL_MEMORY_CHECK_SHA256, 'lab_native_kernels.py': LOCAL_NATIVE_KERNELS_SHA256}
for name, expected_sha256 in LOCAL_FILES.items():
    target = LAB_DIR / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        continue
    source = LAB_SOURCE / name
    if not source.is_file():
        raise FileNotFoundError(f'Missing local lab helper: {source}; apply the lab changes to the checkout first')
    content = source.read_bytes()
    assert hashlib.sha256(content).hexdigest() == expected_sha256, f'Unexpected source: {name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
    print('Copied verified local helper:', name, flush=True)
sys.path.insert(0, str(LAB_DIR))
# Refresh helpers when upgrading an existing notebook; keep checkpoints/logs.
if 'runtime' in globals():
    runtime.stop()
import importlib
importlib.invalidate_caches()
for name in ['lab_memory_check', 'lab_native_kernels', 'lab_runtime', 'cloud_runtime', 'training_monitor', 'lora_recovery', 'checkpoint_backup', 'optimized_training', 'training_stages', 'planner_data', 'pacman_lab', 'api_client']:
    sys.modules.pop(name, None)
from cloud_runtime import CloudRuntime
from lora_recovery import latest_snapshot
from training_stages import specifications, inspect_checkpoint, restore_checkpoint, backup_checkpoint
from pacman_lab import *
from planner_data import prepare_dataset, planner_rollout, PREFIX, RECIPE
from api_client import call, distribution
ensure_node(LAB_DIR)
from lab_runtime import LabRuntime
runtime = LabRuntime(LAB_DIR, training_profile=TRAINING_PROFILE)
print('Jupyter Python:', sys.executable)
print('Training Python:', runtime.python)
print('Physical GPU:', GPU)
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
STAGES = specifications()
manifest = prepare_dataset(LAB_DIR / 'data')
GAME = notebook_game()
STAGE_OWNERS = {}
print('Prepared classic arcade engine, four ghosts and episode-disjoint player snapshots.')

Jupyter Python: /chronos_data/conda_envs/qplus-jupyter-py313/bin/python
Training Python: /chronos_data/conda_envs/qplus-kev-training-py313/bin/python
Physical GPU: 1
Prepared classic arcade engine, four ghosts and episode-disjoint player snapshots.


## 训练曲线

启动单元格从本地 checkout 复制并校验 21 个课程 helper/数据/游戏文件及三个本地运行 helper（`lab_runtime.py`、`lab_memory_check.py`、`lab_native_kernels.py`），保留已有数据、checkpoint 和日志。原课程 commit、模型 revision、数据和内核 SHA-256 保持固定。训练安装仍需要联网下载 Kev、锁定依赖、模型权重和训练数据。

在实验室终端运行 TensorBoard，并从本地 SSH 转发 6006：

```bash
conda activate /chronos_data/conda_envs/qplus-jupyter-py313
tensorboard --logdir /chronos_data/huixu/qpluslearning-runtime-a6000/logs --host 127.0.0.1 --port 6006
```

`logs/<stage>/<attempt>` 保存每步 CE、学习率、梯度、耗时、显存、TensorBoard events，以及原始日志和失败批次形状。

In [2]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'tensorboard==2.20.0'])
LOG_DIR = LAB_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)
SHOW_TENSORBOARD = False  # Run TensorBoard in a lab terminal and forward port 6006
if SHOW_TENSORBOARD:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'tensorboard')
        ip.run_line_magic('tensorboard', f'--logdir "{LOG_DIR}" --reload_interval 5')
    else:
        print('Outside a notebook, run: tensorboard --logdir', LOG_DIR)

In [3]:
runtime.setup()

Tue Oct  6 03:09:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:17:00.0 Off |                  Off |
| 30%   30C    P8             23W /  300W |   46395MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Uninstalled 6 packages in 109ms
 - causal-conv1d==1.7.0 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/native-kernels/96a79f6594e22b1c19fff4914211197b762b1a6ceb6332b6062db4b6aa67a782/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl)
 - einops==0.8.1 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/kernel-wheels/einops-0.8.1-py3-none-any.whl)
 - fla-core==0.5.2 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/kernel-wheels/fla_core-0.5.2-py3-none-any.whl)
 - flash-linear-attention==0.5.2 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/kernel-wheels/flash_linear_attention-0.5.2-py3-none-any.whl)
 - ninja==1.13.0 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/kernel-wheels/ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl)
 - wheel==0.45.1
Using Python 3.13.16 environment at: /chronos_data/conda_envs/qplus-kev-training-py313
Resolved 5 packages in 6ms
Installed 5 packages in 42ms
 + causal-conv1d==1.7.0+cu

Native optimized-kernel import failed:
QPLUS_NATIVE_PLATFORM {"python": "3.13", "prefix": "/chronos_data/conda_envs/qplus-kev-training-py313", "system": "Linux", "machine": "x86_64", "libc": ["glibc", "2.31"], "torch": "2.8.0+cu128", "cuda": "12.8", "cxx11_abi": true, "cuda_home": "/chronos_data/conda_envs/qplus-jupyter-py313", "build_tools": {"setuptools": "84.0.0", "wheel": null, "packaging": "26.3"}}

Traceback (most recent call last):
  File "<string>", line 22, in <module>
    import causal_conv1d, causal_conv1d_cuda
  File "/chronos_data/conda_envs/qplus-kev-training-py313/lib/python3.13/site-packages/causal_conv1d/__init__.py", line 3, in <module>
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn, causal_conv1d_update
  File "/chronos_data/conda_envs/qplus-kev-training-py313/lib/python3.13/site-packages/causal_conv1d/causal_conv1d_interface.py", line 6, in <module>
    from causal_conv1d.cpp_functions import causal_conv1d_fwd_function, causal_conv1d_bwd_func

Using Python 3.13.16 environment at: /chronos_data/conda_envs/qplus-kev-training-py313
Resolved 1 package in 115ms
Installed 1 package in 35ms
 + wheel==0.45.1
Using Python 3.13.16 environment at: /chronos_data/conda_envs/qplus-kev-training-py313
Resolved 1 package in 3ms
Prepared 1 package in 0.61ms
Uninstalled 1 package in 3ms
Installed 1 package in 28ms
 - causal-conv1d==1.7.0 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/kernel-wheels/causal_conv1d-1.7.0+cu12torch2.8cxx11abiTRUE-cp313-cp313-linux_x86_64.whl)
 + causal-conv1d==1.7.0 (from file:///chronos_data/huixu/qpluslearning-runtime-a6000/native-kernels/96a79f6594e22b1c19fff4914211197b762b1a6ceb6332b6062db4b6aa67a782/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl)


{
  "name": "NVIDIA RTX A6000",
  "memory_gib": 47.41253662109375,
  "compute_capability": [
    8,
    6
  ],
  "torch": "2.8.0+cu128",
  "cuda": "12.8",
  "bf16_supported": true,
  "kernel_backward": "passed",
  "optional_packages_before_overlay": {
    "causal_conv1d": false,
    "fla": false
  }
}
Prepared pinned Python 3.13 Kev environment at /chronos_data/conda_envs/qplus-kev-training-py313 using bf16.


## 可选：重新检查本机优化内核

新版 `runtime.setup()` 已自动验证和修复本机卷积扩展，并缓存编译产物。下面的单元格只重新检查绑定；已通过时不会重复编译。保留 `RUN_TRAINING_PREFLIGHT=True`，随后运行真实 CUDA 预检。

In [ ]:
from lab_native_kernels import ensure_native_kernels

native_kernel_recheck = ensure_native_kernels(
    runtime.python, runtime.repo, runtime.workspace, runtime.environment()
)
print(json.dumps(native_kernel_recheck, indent=2))


## 校验锁定的优化训练环境

训练使用 Python 3.13、Torch 2.8.0/CUDA 12.8、Triton 3.4.0、Transformers 5.17.0 和锁定 PEFT。FLA/fla-core 0.5.2、causal-conv1d 1.7.0、einops 0.8.1、Ninja 1.13.0 的 wheel 均校验 SHA-256，并用 `--no-deps` 安装。CUDA convolution wheel 要求 Linux x86_64、Python 3.13 和 CXX11 ABI。

若原 causal-conv1d wheel 导入失败，且诊断明确是系统缺少它要求的 GLIBC 版本，`lab_native_kernels.py` 才用 SHA-256 固定的官方 1.7.0 源码和已有 CUDA 12.8 `nvcc` 本地编译，默认两条编译任务。它在独立训练环境中安装并重新验证 convolution/FLA bindings；保留 Torch 锁定版本、系统 GLIBC、原 wheel checksum 和源码 license。其他导入错误会保留原诊断并停止，不会触发这种重编译。`lab-native-kernels.json` 记录使用原 wheel 还是已验证的本地构建；本地 wheel 按工具链指纹和 checksum 缓存复用。

下方必须保持 `RUN_TRAINING_PREFLIGHT=True`：下载并审计实际 4B 基座后，在 A6000 上用两条真实记录检查 loss、backward、有限 LoRA/head 梯度和 fused AdamW/FLA/convolution 调用。20 分钟是 timeout，不是预计耗时。

开始各阶段完整训练前，另用该阶段实际参数和训练数据扫描 tokenized forward-pass 形状，选择观测到最昂贵的样本，并考虑符合条件的 none-pair siblings。临时模型连续累积八个 microbatch 后更新，重复两次，记录 allocated/reserved 峰值和实际 pass 形状；这不是正式 checkpoint。`logs/<stage>/memory-check-*/memory-preflight.json` 保存检查结果。检查 timeout 为一小时，不是预计耗时。失败不开始完整阶段；通过也不能保证整个阶段所有形状和后续共享 GPU 占用都能适配。

In [4]:
RUN_TRAINING_PREFLIGHT = True  # Required kernel smoke check; each stage also tests long samples before full training.
audit = runtime.prepare_training(run_training_preflight=RUN_TRAINING_PREFLIGHT)
print('Training suite records:', audit['suite_records'])
print('Trainable parameters:', sum(audit['trainable_parameters'].values()))
print('Optimized training verification:', json.dumps(runtime.training_preflight, indent=2))

Loading weights: 100%|██████████| 426/426 [00:01<00:00, 381.15it/s]


Prepared 12576 verified training records and base weights. LoRA/head audit saved.
Running optional two-record Kev loss/backward check (no CUDA profiler)
[kernel-preflight] Starting. Running CUDA training verification. Logs: /chronos_data/huixu/qpluslearning-runtime-a6000/logs/kernel-preflight/20261006-031035-28cdb044
[kernel-preflight] importing pinned Kev/Transformers training stack
Selected optimized training kernels: {"kernels": {"delta_rule": "fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule", "convolution": "causal_conv1d.causal_conv1d_interface.causal_conv1d_fn"}, "versions": {"flash-linear-attention": "0.5.2", "fla-core": "0.5.2", "einops": "0.8.1", "causal-conv1d": "1.7.0", "ninja": "1.13.0", "torch": "2.8.0", "triton": "3.4.0", "transformers": "5.17.0", "peft": "0.21.0"}}
Loading the actual Kev backbone/LoRA/head for CUDA training verification
[kernel-preflight] loading tokenizer and actual backbone/LoRA/head

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 635.55it/

In [5]:
import subprocess

code = """
from huggingface_hub.constants import HF_HUB_CACHE
print("实际模型缓存：", HF_HUB_CACHE, flush=True)
import torch
import causal_conv1d
import causal_conv1d_cuda
print("convolution 内核导入成功")
"""

result = subprocess.run(
    [str(runtime.python), "-c", code],
    cwd=runtime.repo,
    env=runtime.environment(),
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)

实际模型缓存： /chronos_data/transformers_cache/
convolution 内核导入成功




## 持久存储与恢复

实验室 execution 使用 batch 1 × accumulation 8、gradient checkpointing 和 row_budget 2048；保留原模型、数据、epochs、有效 batch 八和 optimizer-step 数。分组、dropout 与优化内核会改变数值轨迹。长于 row budget 的单个问题仍完整运行，不会静默截断。

`SAVE_TO_DRIVE=False`；默认把验证过的恢复 ZIP 保存到 `LAB_DIR/backups`（可用 `QPLUS_BACKUP_DIR` 指定）。这不依赖 Google Drive，同盘副本不能代替异地备份。保留 step 1、每 100 步/五分钟、最终步的完整 LoRA/head、optimizer、scheduler、RNG 和输入数据。checkpoint 与 recovery 路径保持固定；已有文件不会作为新训练被删除。完整输出直接复用，部分输出从恢复快照继续。

Stage 1 支持显式改变 batch/accum/row_budget 的恢复；其他阶段恢复要求原参数一致。每次训练仍有 180 分钟 attempt cap，超时后保留恢复结果；这不是整个阶段的预计耗时。

In [6]:
SAVE_TO_DRIVE = False  # True enables automatic backup and restore; no manual file copying
RESTORE_RECOVERY_ARCHIVE = None  # ZIP from the inspection/export cell; restore into an absent root
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
DRIVE_BACKUP_ROOT = Path(os.environ.get('QPLUS_BACKUP_DIR', str(LAB_DIR / 'backups')))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman/backups')
if RESTORE_RECOVERY_ARCHIVE is not None:
    restore_checkpoint(RESTORE_RECOVERY_ARCHIVE, CHECKPOINT_ROOT)
else:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print('Automatic lab archive backup:', json.dumps(runtime.configure_backup(DRIVE_BACKUP_ROOT, CHECKPOINT_ROOT), indent=2))
print('Later-stage training profile:', runtime.training_profile, 'overrides:', runtime.selected_recipe({}))
print('Stage 1 selects batch 1 x accumulation 8 / row_budget 2048 in its own training cell.')
print('Checkpoints and recovery:', CHECKPOINT_ROOT)

Automatic lab archive backup: {
  "enabled": true,
  "restored": [],
  "backed_up": [],
  "backup_root": "/chronos_data/huixu/qpluslearning-runtime-a6000/backups"
}
Later-stage training profile: memory_safe overrides: {'batch': 1, 'accum': 8, 'checkpointing': 1, 'row_budget': 2048}
Stage 1 selects batch 1 x accumulation 8 / row_budget 2048 in its own training cell.
Checkpoints and recovery: /chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints


### Inspect a failed run / export recovery

This cell is safe to rerun after a training exception. It checks files rather than assuming a checkpoint exists. Select the failed stage's output in `INSPECT_OUTPUT`; choose the original `decision-v7-initial` path to inspect the earlier OOM. Set `EXPORT_RECOVERY=True` to download the output plus its recovery snapshots. Download logs separately before disconnecting. A ZIP containing only configuration files cannot recover model weights.

In [7]:
INSPECT_OUTPUT = CHECKPOINT_ROOT / 'kev-4b-initial'
EXPORT_RECOVERY = False
EXPORT_LOGS = False
def show_training_state(output):
    output = Path(output)
    weights = [output / 'head.pt', *output.glob('adapter_model.*')]
    print('Output:', output, 'exists:', output.exists())
    print('Weight files:', {p.name: p.stat().st_size for p in weights if p.is_file()})
    print('Completed metrics:', (output / 'training_metrics.json').is_file())
    pointer = Path(str(output) + '-recovery') / 'latest.json'
    print('Latest recovery:', json.loads(pointer.read_text()) if pointer.is_file() else 'none')
show_training_state(LAB_DIR / 'checkpoints/decision-v7-initial')
show_training_state(INSPECT_OUTPUT)
archives = []
if EXPORT_RECOVERY:
    import zipfile
    archive = LAB_DIR.parent / 'pacman-stage-recovery.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as saved:
        for folder in [INSPECT_OUTPUT, Path(str(INSPECT_OUTPUT) + '-recovery')]:
            for file in folder.rglob('*'):
                if file.is_file():
                    saved.write(file, file.relative_to(INSPECT_OUTPUT.parent))
    archives.append(archive)
if EXPORT_LOGS:
    archives.append(backup_checkpoint(LOG_DIR, LAB_DIR.parent / 'pacman-training-logs.zip'))
for archive in archives:
    try:
        from google.colab import files
    except ImportError:
        print('Save:', archive)
    else:
        files.download(str(archive))

Output: /chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/decision-v7-initial exists: False
Weight files: {}
Completed metrics: False
Latest recovery: none
Output: /chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-initial exists: False
Weight files: {}
Completed metrics: False
Latest recovery: none


In [8]:
assert 'runtime' in globals(), '请使用原来仍在运行的 kernel'
import importlib, lab_runtime

importlib.reload(lab_runtime)
runtime.__class__ = lab_runtime.LabRuntime
LOCAL_RUNTIME_SHA256 = 'c4bda5336d1b6209a1e95c207b3c4391dc0f913c080c3c6275ad7adb50ce37b4'
LOCAL_FILES['lab_runtime.py'] = LOCAL_RUNTIME_SHA256
print('保留的训练预检状态:', runtime.training_preflight['result'])

保留的训练预检状态: passed


## Stage 1 — Initial decision training (prework)

Only this stage starts fresh LoRA adapters and a pointer head over the pretrained Qwen base. It uses the selected published `experiments/q35-4b-s23.json` seed-2 recipe: all 12,576 `decision-v7` training records, two epochs, batch 4, accumulation 2, gradient checkpointing, learning rate 5e-5, rank 16 / alpha 32, a 256-dimensional head and BF16 autocast over FP32 weights. Option shuffling, none/distractor insertion and 25% none minimal pairs remain active. There is no `--init_from`.

The initial cell performs this stage only. It explicitly sets `INITIAL_EXECUTION={'batch': 4, 'accum': 2, 'row_budget': 0}` before constructing the command; gradient checkpointing remains enabled. The generic memory profile applies to later stages. **By default it resumes a complete recovery snapshot when one exists locally or was restored from Drive; with no checkpoint it starts fresh.** You can select `latest` or a specific saved snapshot with `RESUME_CHECKPOINT`; an explicitly requested missing checkpoint stops the run. A completed initial checkpoint is reused. It streams progress, saves recovery snapshots and writes TensorBoard curves. The full run is prework with a configurable 180-minute attempt cap, not a promised duration. To restore a completed initial checkpoint manually, upload its ZIP, set `RESTORE_ARCHIVE` and declare learner/instructor ownership.

## Stage 1：初始决策训练

使用固定 Qwen3.5-4B-Base 和完整 decision-v7 train，训练两个 epochs。实验室单元格明确选择 `INITIAL_EXECUTION={'batch':1,'accum':8,'row_budget':2048}`。默认有完整快照时恢复，没有快照时新建；完整 checkpoint 复用。`RESUME_CHECKPOINT='latest'` 或完整快照路径可以明确要求恢复，缺失时停止，避免无意从头训练。

In [9]:
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
INITIAL_EXECUTION = {'batch': 1, 'accum': 8, 'row_budget': 2048}  # A6000 execution; effective batch remains eight
runtime.initial_execution = dict(INITIAL_EXECUTION)
print('Selected initial execution:', runtime.initial_execution, flush=True)
print('Selected command:', runtime.pretraining_command(INITIAL), flush=True)
from lora_recovery import latest_snapshot
RESUME_CHECKPOINT = 'latest' if latest_snapshot(Path(str(INITIAL) + '-recovery')) else None
# To choose a saved step, replace the line above with its full snapshot directory path.
RESTORE_ARCHIVE = None  # Example: '/content/pacman-initial-checkpoint.zip'
RESTORED_CHECKPOINT_OWNER = 'learner'  # 'instructor' for a supplied fallback
if RESTORE_ARCHIVE is None:
    if (INITIAL / 'run-evidence.json').is_file():
        print('Using completed initial checkpoint:', INITIAL)
    else:
        runtime.pretrain(INITIAL, steps=0, resume_from=RESUME_CHECKPOINT,
                         allow_execution_change=RESUME_CHECKPOINT is not None)
    STAGE_OWNERS['initial'] = 'learner'
else:
    restore_checkpoint(RESTORE_ARCHIVE, INITIAL)
    STAGE_OWNERS['initial'] = RESTORED_CHECKPOINT_OWNER
published = json.loads((runtime.repo / 'experiments/q35-4b-s23.json').read_text())[0]
initial_config, initial_metrics = inspect_checkpoint(INITIAL, stage='initial', owner=STAGE_OWNERS['initial'], recipe=runtime.selected_initial_recipe(published))

Selected initial execution: {'batch': 1, 'accum': 8, 'row_budget': 2048}
Selected command: ['/chronos_data/conda_envs/qplus-kev-training-py313/bin/python', '-m', 'kev.train', '--suite', 'evals/v7/decision-v7', '--out', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-initial', '--device', 'cuda', '--lora', '16', '--lora_targets', 'all', '--head_dim', '256', '--weights_dtype', 'fp32', '--epochs', '2', '--batch', '1', '--accum', '8', '--dtype', 'bf16', '--checkpointing', '1', '--p_none_pair', '0.25', '--lr', '5e-05', '--base', 'Qwen/Qwen3.5-4B-Base', '--base_revision', '1001bb4d826a52d1f399e183466143f4da7b741b', '--seed', '2', '--row_budget', '2048']
Checking initial long-sample training memory before the full stage. This is a measured probe, not a guarantee for every later batch.
[initial-memory-check] Starting. Running CUDA training verification. Logs: /chronos_data/huixu/qpluslearning-runtime-a6000/logs/initial/memory-check-20261006-031321-d5b166b8
[initial-memory-c

TimeoutError: initial exceeded 180 minutes; logs kept at /chronos_data/huixu/qpluslearning-runtime-a6000/logs/initial/20261006-031504-cb8aac48

In [10]:
runtime.pretrain(
    INITIAL,
    steps=0,
    timeout_minutes=360,
    resume_from='latest',
    allow_execution_change=True,
)

Checking initial long-sample training memory before the full stage. This is a measured probe, not a guarantee for every later batch.
[initial-memory-check] Starting. Running CUDA training verification. Logs: /chronos_data/huixu/qpluslearning-runtime-a6000/logs/initial/memory-check-20261006-095820-bde71f62
[initial-memory-check] verifying the required pinned optimized CUDA bindings
[initial-memory-check] loading the stage tokenizer and verified data, including its replay records
[initial-memory-check] scanning actual encoded pass shapes across the configured training epochs
[initial-memory-check] scanned 500 tokenized cases; current pass cost 312
[initial-memory-check] scanned 1000 tokenized cases; current pass cost 627
[initial-memory-check] scanned 1500 tokenized cases; current pass cost 89
[initial-memory-check] scanned 2000 tokenized cases; current pass cost 128
[initial-memory-check] scanned 2500 tokenized cases; current pass cost 152
[initial-memory-check] 0.2 min elapsed; scanned

PosixPath('/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-initial')

In [11]:
PREWORK_ARCHIVE = backup_checkpoint(INITIAL, LAB_DIR.parent / 'pacman-initial-checkpoint.zip')
try:
    from google.colab import files
except ImportError:
    print('Save initial checkpoint:', PREWORK_ARCHIVE)
else:
    files.download(str(PREWORK_ARCHIVE))

Save initial checkpoint: /chronos_data/huixu/pacman-initial-checkpoint.zip


## Prepare the intermediate-stage data (prework)

Verify dates/missing-evidence JSONL and the separate `documents-v1` train partition. Reconstruct skills/devtools by concatenating verified `hard-v1` train then `devtools-v1` train. Each input matches its published checksum and count. The course pins the combined checksum; the unavailable historical round-10 concatenation is not claimed byte-identical. Replay always samples `decision-v7` **train**. Development/test rows do not enter any training stage.

In [12]:
runtime.prepare_intermediate_data()
STAGES = specifications()
print({name: {'new_records': stage['records'], 'replay_records': stage['replay']} for name, stage in STAGES.items()})

Dates/missing-evidence data verified: 1,425 records


fetched documents-v1/train.jsonl from jaredpalmer/kev-suites@cc4bac803e
Documents data verified: 5,219 records
fetched hard-v1/train.jsonl from jaredpalmer/kev-suites@cc4bac803e
Verified training partition: evals/hard-v1
Verified training partition: evals/devtools-v1
Skills/devtools data verified: 11,320 records (course-pinned concatenation)
{'dates': {'new_records': 1425, 'replay_records': 2000}, 'documents': {'new_records': 5219, 'replay_records': 2000}, 'skills': {'new_records': 11320, 'replay_records': 4000}}


## Stage 2 — Dates and missing evidence (prework)

Warm-start from your **completed Stage 1** checkpoint. On a fresh runtime without a restored checkpoint, run Stage 1 to completion first; setup/model downloads do not create this trained checkpoint. Use 1,425 generated records (900 date-policy cases, 255 missing-fact cases and 270 intact controls) plus 2,000 replayed `decision-v7` training records. Published settings: one epoch, learning rate 2e-5, batch 4, accumulation 2, gradient checkpointing, BF16, seed 1 and 25% none minimal pairs. This is a separate run and a separate checkpoint. Its 180-minute attempt cap is a scheduling limit pending GPU measurements.

In [13]:
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
from lora_recovery import latest_snapshot
STAGES = specifications()
RESUME_DATES = latest_snapshot(Path(str(DATES) + '-recovery')) is not None
RESTORE_DATES_ARCHIVE = None
RESTORED_DATES_OWNER = 'learner'
if RESTORE_DATES_ARCHIVE is None:
    if (DATES / 'run-evidence.json').is_file():
        print('Using completed dates checkpoint:', DATES)
    else:
        print('Dates command:', runtime.intermediate_command('dates', DATES, INITIAL), flush=True)
        runtime.intermediate('dates', DATES, init_from=INITIAL, resume=RESUME_DATES)
    STAGE_OWNERS['dates'] = 'learner'
else:
    restore_checkpoint(RESTORE_DATES_ARCHIVE, DATES)
    STAGE_OWNERS['dates'] = RESTORED_DATES_OWNER
dates_config, dates_metrics = inspect_checkpoint(DATES, stage='dates', owner=STAGE_OWNERS['dates'], parent=INITIAL, recipe=runtime.selected_recipe(STAGES['dates']['args']), data_spec=STAGES['dates'])

Dates command: ['/chronos_data/conda_envs/qplus-kev-training-py313/bin/python', '-m', 'kev.train', '--init_from', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-initial', '--data', '/chronos_data/huixu/qpluslearning-runtime-a6000/kev/evals/night2/dates_unknowable.jsonl', '--out', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-dates', '--base', 'Qwen/Qwen3.5-4B-Base', '--lora', '16', '--head_dim', '256', '--option_isolation', '0', '--special_embeddings', '0', '--weights_dtype', 'fp32', '--dtype', 'bf16', '--device', 'cuda', '--epochs', '1', '--lr', '2e-05', '--batch', '1', '--accum', '8', '--max_steps', '0', '--max_state', '384', '--checkpointing', '1', '--p_none', '0.1', '--p_none_distract', '0.12', '--p_distract', '0.15', '--seed', '1', '--lora_targets', 'all', '--base_revision', '1001bb4d826a52d1f399e183466143f4da7b741b', '--row_budget', '2048', '--p_none_pair', '0.25', '--replay', '2000', '--suite', 'evals/v7/decision-v7']
Checking dates lon

In [14]:
DATES_ARCHIVE = backup_checkpoint(DATES, LAB_DIR.parent / 'pacman-dates-checkpoint.zip')
try:
    from google.colab import files
except ImportError:
    print('Save dates checkpoint:', DATES_ARCHIVE)
else:
    files.download(str(DATES_ARCHIVE))

Save dates checkpoint: /chronos_data/huixu/pacman-dates-checkpoint.zip


## Stage 3 — Documents (prework)

Warm-start from **Stage 2**. If you restored a completed dates checkpoint, run runtime setup, optimized preparation, storage and intermediate-data preparation, then start here. You can skip the Stage 1/2 training cells; recovery files are not needed to initialize documents. Set `DATES` to the restored folder below. Earlier stage archives that are unavailable are reported as missing in the comparison/export.

Use 5,219 consumer-finance complaint records plus 2,000 replayed decision-v7 training records. Published settings: one epoch, learning rate 2e-5, batch 2 × accumulation 4, BF16, FP32 stored weights, state budget 7,552, gradient checkpointing, seed 2 and 25% none minimal pairs. Expect 903 optimizer steps. The memory profile uses batch 1 × accumulation 8 with bounded row passes. Save this checkpoint before continuing; documents and skills are separate in Kev-4B.

In [15]:
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
from lora_recovery import latest_snapshot
STAGES = specifications()
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'  # Set this to your completed dates folder
STAGE_OWNERS.setdefault('dates', 'learner')  # 'instructor' for a supplied checkpoint
RESUME_DOCUMENTS = latest_snapshot(Path(str(DOCUMENTS) + '-recovery')) is not None
RESTORE_DOCUMENTS_ARCHIVE = None
RESTORED_DOCUMENTS_OWNER = 'learner'
if RESTORE_DOCUMENTS_ARCHIVE is None:
    if (DOCUMENTS / 'run-evidence.json').is_file():
        print('Using completed documents checkpoint:', DOCUMENTS)
    else:
        print('Documents command:', runtime.intermediate_command('documents', DOCUMENTS, DATES), flush=True)
        runtime.intermediate('documents', DOCUMENTS, init_from=DATES, resume=RESUME_DOCUMENTS)
    STAGE_OWNERS['documents'] = 'learner'
else:
    restore_checkpoint(RESTORE_DOCUMENTS_ARCHIVE, DOCUMENTS)
    STAGE_OWNERS['documents'] = RESTORED_DOCUMENTS_OWNER
documents_config, documents_metrics = inspect_checkpoint(DOCUMENTS, stage='documents', owner=STAGE_OWNERS['documents'], parent=DATES, recipe=runtime.selected_recipe(STAGES['documents']['args']), data_spec=STAGES['documents'])

Documents command: ['/chronos_data/conda_envs/qplus-kev-training-py313/bin/python', '-m', 'kev.train', '--init_from', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-dates', '--data', '/chronos_data/huixu/qpluslearning-runtime-a6000/kev/evals/documents-v1/train.jsonl', '--out', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-documents', '--base', 'Qwen/Qwen3.5-4B-Base', '--lora', '16', '--head_dim', '256', '--option_isolation', '0', '--special_embeddings', '0', '--weights_dtype', 'fp32', '--dtype', 'bf16', '--device', 'cuda', '--epochs', '1', '--lr', '2e-05', '--batch', '1', '--accum', '8', '--max_steps', '0', '--max_state', '7552', '--checkpointing', '1', '--p_none', '0.1', '--p_none_distract', '0.12', '--p_distract', '0.15', '--seed', '2', '--lora_targets', 'all', '--base_revision', '1001bb4d826a52d1f399e183466143f4da7b741b', '--row_budget', '2048', '--replay', '2000', '--p_none_pair', '0.25', '--suite', 'evals/v7/decision-v7']
Checking documen

In [16]:
DOCUMENTS_ARCHIVE = backup_checkpoint(DOCUMENTS, LAB_DIR.parent / 'pacman-documents-checkpoint.zip')
try:
    from google.colab import files
except ImportError:
    print('Save documents checkpoint:', DOCUMENTS_ARCHIVE)
else:
    files.download(str(DOCUMENTS_ARCHIVE))

Save documents checkpoint: /chronos_data/huixu/pacman-documents-checkpoint.zip


## Stage 4 — Skills and developer tools (prework)

Warm-start from **Stage 3 documents**. Use 6,000 generated skill records plus 5,320 developer-tooling records, with 4,000 replayed decision-v7 records. Published settings: one epoch, learning rate 2e-5, batch 2 × accumulation 4, BF16, FP32 stored weights, state budget 7,552, gradient checkpointing, seed 1 and 25% none minimal pairs. Expect 1,915 optimizer steps. The memory profile uses batch 1 × accumulation 8. This checkpoint is the class baseline. Complete all four general stages before class; their configurable 180-minute attempt caps are scheduling limits, not timing estimates.

A 4B model still uses hybrid DeltaNet/full attention and needs optimized kernels. Start fresh from the pinned 4B base; 0.8B adapters cannot initialize this curriculum. Published H100/H200 timings do not predict this GPU's duration.

In [17]:
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
from lora_recovery import latest_snapshot
STAGES = specifications()
RESUME_SKILLS = latest_snapshot(Path(str(SKILLS) + '-recovery')) is not None
RESTORE_SKILLS_ARCHIVE = None
RESTORED_SKILLS_OWNER = 'learner'
if RESTORE_SKILLS_ARCHIVE is None:
    if (SKILLS / 'run-evidence.json').is_file():
        print('Using completed skills checkpoint:', SKILLS)
    else:
        print('Skills and developer tools command:', runtime.intermediate_command('skills', SKILLS, DOCUMENTS), flush=True)
        runtime.intermediate('skills', SKILLS, init_from=DOCUMENTS, resume=RESUME_SKILLS)
    STAGE_OWNERS['skills'] = 'learner'
else:
    restore_checkpoint(RESTORE_SKILLS_ARCHIVE, SKILLS)
    STAGE_OWNERS['skills'] = RESTORED_SKILLS_OWNER
skills_config, skills_metrics = inspect_checkpoint(SKILLS, stage='skills', owner=STAGE_OWNERS['skills'], parent=DOCUMENTS, recipe=runtime.selected_recipe(STAGES['skills']['args']), data_spec=STAGES['skills'])

Skills and developer tools command: ['/chronos_data/conda_envs/qplus-kev-training-py313/bin/python', '-m', 'kev.train', '--init_from', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-documents', '--data', '/chronos_data/huixu/qpluslearning-runtime-a6000/kev/evals/lab-kev4b/skills/train.jsonl', '--out', '/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-skills', '--base', 'Qwen/Qwen3.5-4B-Base', '--lora', '16', '--head_dim', '256', '--option_isolation', '0', '--special_embeddings', '0', '--weights_dtype', 'fp32', '--dtype', 'bf16', '--device', 'cuda', '--epochs', '1', '--lr', '2e-05', '--batch', '1', '--accum', '8', '--max_steps', '0', '--max_state', '7552', '--checkpointing', '1', '--p_none', '0.1', '--p_none_distract', '0.12', '--p_distract', '0.15', '--seed', '1', '--lora_targets', 'all', '--base_revision', '1001bb4d826a52d1f399e183466143f4da7b741b', '--row_budget', '2048', '--replay', '4000', '--p_none_pair', '0.25', '--suite', 'evals/v7/decision

TimeoutError: skills exceeded 180 minutes; logs kept at /chronos_data/huixu/qpluslearning-runtime-a6000/logs/skills/20261006-154131-51ac02de

In [18]:
runtime.intermediate(
    'skills',
    SKILLS,
    init_from=DOCUMENTS,
    timeout_minutes=360,
    resume=True,
)

# 完成后补齐原单元格因超时而未执行的部分
STAGE_OWNERS['skills'] = 'learner'
skills_config, skills_metrics = inspect_checkpoint(
    SKILLS,
    stage='skills',
    owner=STAGE_OWNERS['skills'],
    parent=DOCUMENTS,
    recipe=runtime.selected_recipe(STAGES['skills']['args']),
    data_spec=STAGES['skills'],
)

Checking skills long-sample training memory before the full stage. This is a measured probe, not a guarantee for every later batch.
[skills-memory-check] Starting. Running CUDA training verification. Logs: /chronos_data/huixu/qpluslearning-runtime-a6000/logs/skills/memory-check-20261007-012041-727eac9f
[skills-memory-check] verifying the required pinned optimized CUDA bindings
[skills-memory-check] loading the stage tokenizer and verified data, including its replay records
replay: 4000 of 12576 suite training records mixed with 11320 from /chronos_data/huixu/qpluslearning-runtime-a6000/kev/evals/lab-kev4b/skills/train.jsonl
[skills-memory-check] 0.2 min elapsed; loading the stage tokenizer and verified data, including its replay records
[skills-memory-check] scanning actual encoded pass shapes across the configured training epochs
[skills-memory-check] scanned 500 tokenized cases; current pass cost 648
[skills-memory-check] scanned 1000 tokenized cases; current pass cost 496
[skills-me

In [19]:
SKILLS_ARCHIVE = backup_checkpoint(SKILLS, LAB_DIR.parent / 'pacman-skills-checkpoint.zip')
try:
    from google.colab import files
except ImportError:
    print('Save skills checkpoint:', SKILLS_ARCHIVE)
else:
    files.download(str(SKILLS_ARCHIVE))

Save skills checkpoint: /chronos_data/huixu/pacman-skills-checkpoint.zip


## 加载已完成的 Skills checkpoint

在新 kernel 中先运行启动、`runtime.setup()`、优化环境准备和存储单元格。如果没有完整 Skills checkpoint，按顺序完成 Stage 1–4；下载基座不会产生已训练的 Skills adapter。

如果已有完整 Skills checkpoint，将 native 文件放在 `LAB_DIR/checkpoints/kev-4b-skills`，或恢复完整自动备份。下方单元格检查 stage、文件和完整步数，加载训练过的 LoRA/head，并明确报告缺失的早期阶段 archive。保留 learner/instructor ownership。

In [20]:
# Load completed Skills checkpoint for the lab
from pathlib import Path
import json
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
GENERAL = SKILLS
required = ['head.pt', 'adapter_config.json', 'training_config.json', 'training_metrics.json', 'run-evidence.json']
missing = [name for name in required if not (GENERAL / name).is_file()]
if missing or not any(file.is_file() for file in GENERAL.glob('adapter_model.*')):
    raise RuntimeError(f'Restore the completed Skills checkpoint to {GENERAL}; missing files: {missing}, or adapter weights.')
STAGE_OWNERS = globals().get('STAGE_OWNERS', {})
STAGE_OWNERS.setdefault('skills', 'learner')  # 'instructor' for a supplied checkpoint
stage_checkpoints = {'initial': INITIAL, 'dates': DATES, 'documents': DOCUMENTS, 'skills': SKILLS}
stage_metrics, stage_configs = {}, {}
for stage, folder in stage_checkpoints.items():
    config_path, metrics_path = folder / 'training_config.json', folder / 'training_metrics.json'
    stage_configs[stage] = json.loads(config_path.read_text()) if config_path.is_file() else None
    stage_metrics[stage] = json.loads(metrics_path.read_text()) if metrics_path.is_file() else None
initial_config, dates_config, documents_config, skills_config = [stage_configs[name] for name in stage_checkpoints]
initial_metrics, dates_metrics, documents_metrics, skills_metrics = [stage_metrics[name] for name in stage_checkpoints]
skills_evidence = json.loads((GENERAL / 'run-evidence.json').read_text())
if (skills_evidence['stage'] != 'skills' or skills_metrics['optimizer_steps'] != 1915
        or skills_metrics['requested_records'] != 15320 or skills_config['args'].get('max_steps', 0)
        or skills_metrics.get('truncated_records', 0) or skills_metrics.get('rejected_records', 0)):
    raise ValueError('Complete the full Skills stage before starting the lab.')
missing_stage_archives = [stage for stage, folder in stage_checkpoints.items()
                         if not (folder / 'head.pt').is_file() or not list(folder.glob('adapter_model.*'))]
print('Earlier stage archives unavailable:', missing_stage_archives)
models = runtime.start(GENERAL)
print(json.dumps(models, indent=2))

Earlier stage archives unavailable: []
Kev ready at notebook-local localhost:8009
Active LoRA/head: {
  "checkpoint": "/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-skills",
  "checkpoint_name": "kev-4b-skills",
  "stage": "skills",
  "optimizer_steps": 1915,
  "checkpoint_sha256": "e653f96f1420f4f2fa41a561d9a0cbd0bc1e37f0aaa83c50c5ab9bff68b4742f",
  "adapter_sha256": "3380cfd50992b5a0431c28fe6f0c832def3fad07d0ffe479a0b3d7fd86892611",
  "base": "Qwen/Qwen3.5-4B-Base",
  "base_revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
  "lora_rank": 16,
  "dtype": "bfloat16",
  "temperature": 1.0,
  "api_alias": "kev-latest",
  "pacman_fine_tuned": false
}
{
  "models": [
    {
      "name": "kev-latest",
      "description": "Kev pointer head on Qwen/Qwen3.5-4B-Base, serving /chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-skills at temperature 1.00",
      "release_date": "2026-10-07",
      "run": "/chronos_data/huixu/qpluslearning-runtime-a6000/che

## Calibration is separate from training

Kev's release fitted one probability temperature after its four training stages. We do not copy that fitted value into freshly trained checkpoints. This notebook keeps their own raw probabilities. A workload calibration experiment needs suitable held-out labels and is outside the mandatory 30-minute Pac-Man fine-tuning block. It does not update LoRA/head weights or change the top-ranked action.

## 可选的交互游戏

原 Kev 浏览器控制使用 Colab kernel callback；普通 Jupyter 中下方单元格会提示跳过，不影响后续真实模型的结构化评估和 rollout。该单元格不会用规则控制器伪装成模型推理。

In [21]:
from IPython.display import display, HTML, JSON
bridge = NotebookBridge(LAB_DIR / 'results/player-trace.jsonl', runtime.active_model_info)
try:
    from google.colab import output
except ImportError:
    print('Kaggle/local: continue with the decision and rollout cells below.')
else:
    output.register_callback('pacman.decide', lambda state: JSON(bridge.decide(state)))
    output.register_callback('pacman.model', lambda: JSON(bridge.model()))
    display(HTML(GAME))

Kaggle/local: continue with the decision and rollout cells below.


## CP1 — Fine-tune and evaluate Kev on Pac-Man game states

Use the committed planning-labelled game states to adapt your completed Skills LoRA/head, then compare the baseline and task adapter on the same held-out boards. This is the lab's only assessed checkpoint.

Complete the data review, full task training and paired evaluation below. Submit the trained adapter/head, reviewed training labels, curves and `comparison.json`, with a short explanation of one changed move or remaining mistake. Explain how the native ghost rules affect a planning label and why the search is approximate. Completion requires the full 762-update recipe, no dropped/truncated records, and recorded before/after measurements; improvement is an experimental result to measure.

### Inspect the planning data (20–30 minutes)

Data has **4,096 training, 256 development and 256 evaluation** snapshots from valid native-engine trajectories, starting at levels 1, 2, 3 and 5. At every labelled board, the teacher simulates up to **20 future player moves** for every legal first action, retaining **eight paths per first action** under **two independent frightened-mode randomness scenarios**. Chase/scatter movement is predicted from each ghost's deterministic native targeting rule, rather than a random walk. Native code handles speeds, timers, release counters, collisions, power pellets, fruit and score.

The objective ranks worst-scenario survival first, then mean `score gained + 5 × pellets collected − 4 × repeat visits − 6 × distance to the next pellet + 2 × min(dangerous-ghost distance, 8)`. The stored candidates show the best sequence found for each direction. **Beam pruning and the finite horizon mean this is an approximate planning teacher, not a globally optimal full-game solver.** The teacher uses current native timers/pixel offsets; the learner sees board, modes, history and elapsed frames, so this is privileged-state imitation. The teacher resamples future RNG rather than looking ahead at the actual episode RNG. Search sequences, labels, replay seeds and scores stay in `_meta`, outside model inputs.

Collection uses 70% planning, 20% the earlier heuristic and 10% random legal moves to include recovery states. Snapshots are spaced four moves apart. Whole episodes and exact observable snapshots are disjoint across splits; all use the classic maze. Inspect the manifest's episode counts, action balance, four-ghost, frightened, fruit and junction coverage, and the CPU teacher-comparison report.

Inspect three training boards and enter your own legal move labels in `EDITS` before checking the teacher. Keep all edits in the training partition. Fix the training file and one candidate before opening evaluation data. The versioned planner files and checkpoint preserve earlier heuristic experiments. Warm-start this candidate from Skills.

In [22]:
training_file = LAB_DIR / 'data/pacman-planner-v1-train-reviewed.jsonl'
label_source = training_file if training_file.is_file() else LAB_DIR / 'data/pacman-planner-v1-train.jsonl'
training = [json.loads(line) for line in label_source.read_text().splitlines()]
print('Training labels:', label_source)
for row in training[:3]:
    print(row['_meta']['id'], json.dumps(row['state'], indent=2))
    print('Legal:', list(row['questions']['move']['criteria']))
EDITS = {}  # Example after inspecting a board: {'train-board-0000': 'left'}
assert set(EDITS).issubset({r['_meta']['id'] for r in training}), 'Unknown training board ID'
for row in training:
    if row['_meta']['id'] in EDITS:
        label = EDITS[row['_meta']['id']]
        assert label in row['questions']['move']['criteria'], 'Choose a legal move'
        row['questions']['move'].update(label=label, src='learner_annotation')
        row['_meta']['label_source'] = 'learner annotation'
if EDITS or not training_file.is_file():
    training_file.write_text(''.join(json.dumps(row) + '\n' for row in training))
print('Teacher labels for inspected boards:', [(r['_meta']['id'], r['questions']['move']['label']) for r in training[:3]])
print('Search alternatives:', training[0]['_meta'].get('teacher', {}).get('candidates', []))
print('Split sizes:', manifest['counts'])
print('Coverage:', manifest['coverage'])
print('CPU teacher comparison:', json.loads((LAB_DIR / 'data/pacman-planner-v1-quality.json').read_text())['summary'])
before_dev = evaluate(LAB_DIR / 'data/pacman-planner-v1-development.jsonl')
print('Development accuracy:', before_dev['accuracy'])

Training labels: /chronos_data/huixu/qpluslearning-runtime-a6000/data/pacman-planner-v1-train.jsonl
train-board-0000 {
  "game": "classic-pacman",
  "engine_revision": "7407174c1d6a38be8cd230577489e39e0873145b",
  "maze": [
    "############################",
    "############################",
    "############################",
    "############################",
    "#............##............#",
    "#.####.#####.##.#####.####.#",
    "#o####.#####.##.#####.####o#",
    "#.####.#####.##.#####.####.#",
    "#..........................#",
    "#.####.##.########.##.####.#",
    "#.####.##.########.##.####.#",
    "#......##....##....##......#",
    "######.##### ## #####.######",
    "######.##### ## #####.######",
    "######.##          ##.######",
    "######.## ######## ##.######",
    "######.## ######## ##.######",
    "      .   ########   .      ",
    "######.## ######## ##.######",
    "######.## ######## ##.######",
    "######.##          ##.######",
    "######.## #####

## CP1：训练 Pac-Man adapter

完整训练 4,096 planning-labelled boards 加 2,000 decision-v7 replay，one epoch、lr 2e-5、rank-16 LoRA、head-256、max_state 4096，共 **762 optimizer updates**。

实验室 execution 明确使用 batch 1 × accumulation 8、row_budget 2048。打印的命令、保存的 training_config、下面的 recipe 一致性检查和最终 evidence 使用同一选择。数据 manifest 保留课程发布 recipe，实际 lab execution 另外记录，不改源数据或删除断言。开始训练前释放推理模型显存；已有 recovery 必须匹配本次参数。

In [23]:
print(json.dumps(training[0], indent=2))
CHECKPOINT = CHECKPOINT_ROOT / 'kev-4b-pacman-planner-v1'
from lora_recovery import latest_snapshot
RESUME_PACMAN = latest_snapshot(Path(str(CHECKPOINT) + '-recovery')) is not None
PACMAN_RECIPE = runtime.selected_pacman_recipe(RECIPE)
print('Pac-Man recipe and lab execution:', PACMAN_RECIPE)
print('Selected Pac-Man command:', runtime.finetuning_command(training_file, CHECKPOINT, GENERAL))
if (CHECKPOINT / 'run-evidence.json').is_file():
    evidence = json.loads((CHECKPOINT / 'run-evidence.json').read_text())
    assert evidence['training_data_sha256'] == hashlib.sha256(training_file.read_bytes()).hexdigest(), 'Saved checkpoint used different labels; select a new checkpoint output to retrain.'
    checkpoint = CHECKPOINT
    print('Using completed Pac-Man checkpoint:', checkpoint)
else:
    checkpoint = runtime.finetune(training_file, CHECKPOINT, init_from=GENERAL, steps=0, resume=RESUME_PACMAN)
metrics = json.loads((checkpoint / 'training_metrics.json').read_text())
print(metrics)
assert len(training) == manifest['counts']['train'], 'Preserve the complete training partition'
saved_recipe = json.loads((checkpoint / 'training_config.json').read_text())['args']
for key, value in PACMAN_RECIPE.items():
    assert saved_recipe[key] == (float(value) if key == 'lr' else value), f'Checkpoint used another recipe: {key}'
assert metrics['optimizer_steps'] == manifest['expected_optimizer_steps'], 'Complete the full planner/replay stage'
assert not metrics.get('truncated_records', 0) and not metrics.get('rejected_records', 0), 'No state truncation or dropped records'
assert metrics['records_seen'] == metrics['requested_records'] == manifest['expected_training_requests'], 'Complete one epoch including replay'

{
  "state": {
    "game": "classic-pacman",
    "engine_revision": "7407174c1d6a38be8cd230577489e39e0873145b",
    "maze": [
      "############################",
      "############################",
      "############################",
      "############################",
      "#............##............#",
      "#.####.#####.##.#####.####.#",
      "#o####.#####.##.#####.####o#",
      "#.####.#####.##.#####.####.#",
      "#..........................#",
      "#.####.##.########.##.####.#",
      "#.####.##.########.##.####.#",
      "#......##....##....##......#",
      "######.##### ## #####.######",
      "######.##### ## #####.######",
      "######.##          ##.######",
      "######.## ######## ##.######",
      "######.## ######## ##.######",
      "      .   ########   .      ",
      "######.## ######## ##.######",
      "######.## ######## ##.######",
      "######.##          ##.######",
      "######.## ######## ##.######",
      "######.## ######## ##.######",


### Evaluate the baseline and task adapter (60–80 minutes)

The candidate is now fixed. Score both models on the same **256 evaluation snapshots** and save predictions by ID. Report strict and tie-aware teacher agreement, lower search-survival choices, search-value regret within the same survival class, and immediate captures. These are comparisons with an approximate teacher, not full-game win rates. Fine-tuning may leave answers unchanged or make them worse.

In [24]:
runtime.start(GENERAL)
before = evaluate(LAB_DIR / 'data/pacman-planner-v1-evaluation.jsonl')
before_identity = runtime.active_model_info()
before_run = rollout(max_turns=128)
runtime.start(checkpoint)
after = evaluate(LAB_DIR / 'data/pacman-planner-v1-evaluation.jsonl')
after_identity = runtime.active_model_info()
after_run = rollout(max_turns=128)
comparison = {'base_model': audit['base'], 'base_revision': audit['base_revision'], 'baseline_checkpoint': str(GENERAL), 'checkpoint_owners': STAGE_OWNERS, 'fine_tuned_checkpoint': str(checkpoint), 'active_adapters': {'before': before_identity, 'after': after_identity}, 'before': before, 'after': after, 'rollouts': {'general': before_run, 'fine_tuned': after_run}, 'general_training_stages': stage_metrics, 'missing_general_stage_archives': missing_stage_archives, 'pacman_training': metrics, 'runtime': runtime.gpu}
comparison['pacman_dataset'] = manifest
comparison['lab_memory_preflights'] = runtime.memory_preflights
comparison['lab_native_kernels'] = runtime.native_training_kernels
comparison['lab_execution'] = {'initial': runtime.initial_execution, 'pacman': PACMAN_RECIPE,
                               'physical_gpu': GPU, 'training_python': str(runtime.python)}
(LAB_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2))
for name, result in [('general', before), ('fine_tuned', after)]:
    print(name, {key:result[key] for key in ['accuracy', 'tie_aware_teacher_accuracy', 'lower_search_survival_choices', 'mean_same_survival_search_regret', 'caught_next_turn']})
for name, episode in [('general', before_run), ('fine_tuned', after_run)]:
    print(name, {key:episode[key] for key in ['turns', 'dots_collected', 'score', 'repeated_tiles', 'outcome']})

Kev ready at notebook-local localhost:8009
Active LoRA/head: {
  "checkpoint": "/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-skills",
  "checkpoint_name": "kev-4b-skills",
  "stage": "skills",
  "optimizer_steps": 1915,
  "checkpoint_sha256": "e653f96f1420f4f2fa41a561d9a0cbd0bc1e37f0aaa83c50c5ab9bff68b4742f",
  "adapter_sha256": "3380cfd50992b5a0431c28fe6f0c832def3fad07d0ffe479a0b3d7fd86892611",
  "base": "Qwen/Qwen3.5-4B-Base",
  "base_revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
  "lora_rank": 16,
  "dtype": "bfloat16",
  "temperature": 1.0,
  "api_alias": "kev-latest",
  "pacman_fine_tuned": false
}
Kev ready at notebook-local localhost:8009
Active LoRA/head: {
  "checkpoint": "/chronos_data/huixu/qpluslearning-runtime-a6000/checkpoints/kev-4b-pacman-planner-v1",
  "checkpoint_name": "kev-4b-pacman-planner-v1",
  "stage": "pacman",
  "optimizer_steps": 762,
  "checkpoint_sha256": "574805e644a6ad9ede8bf8a5ae7812189084a943e3d4fde656f9cd01f546a343",
  "a

## 保存实验室运行结果

导出包含实际评估、已审阅标签、可用的 adapter/head、配置、运行 receipts、训练日志和 TensorBoard events。基础模型和包不进入 submission ZIP。缺少的早期阶段 archive 会明确列出。导出单元格打印本地文件路径；从你本地电脑通过 scp 下载，保留真实 GPU 型号、显存和耗时记录。

In [25]:
runtime.stop()
archive = str(LAB_DIR.parent / 'pacman-lab-submission.zip')
print('Unavailable earlier stage archives:', missing_stage_archives)
# Export small adapters/heads and results, without foundation weights or packages.
import zipfile
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as out:
    for file in [LAB_DIR/'comparison.json', training_file, LAB_DIR/'data/pacman-planner-v1-manifest.json', LAB_DIR/'data/pacman-planner-v1-quality.json', LAB_DIR/'runtime-preflight.json', LAB_DIR/'lab-native-kernels.json', LAB_DIR/'optimized-training-preflight.json', LAB_DIR/'trainable-parameters.json']:
        out.write(file, file.relative_to(LAB_DIR))
    for folder in [INITIAL, DATES, DOCUMENTS, SKILLS, checkpoint]:
        for file in folder.rglob('*'):
            if file.is_file():
                out.write(file, Path('checkpoints') / folder.name / file.relative_to(folder))
    for file in LOG_DIR.rglob('*'):
        if file.is_file():
            out.write(file, file.relative_to(LAB_DIR))
try:
    from google.colab import files
except ImportError:
    print('Download:', archive)
else:
    files.download(archive)

Unavailable earlier stage archives: []
Download: /chronos_data/huixu/pacman-lab-submission.zip
